# Part I — Milestone 1: Data Crawling

## 2. Methodology

### 2.1 Entry discovery and version coverage

Explain how the implementation resolves each assigned base ID, identifies all available versions, and handles missing or invalid IDs. The arXiv API, OAI-PMH, and pre-collected metadata are options described by the specification; do not imply all were implemented unless they were.

**Implementation:** TODO

**Error handling / retry / rate limiting:** TODO

### 2.2 Source download and storage

Explain how each version is downloaded from the version-specific source endpoint and how archives are handled. Only `.tex` and `.bib` files should enter the working corpus; figure binaries and other unnecessary files should be excluded during extraction, while figure commands remain untouched in the TeX text.

**Directory convention:** TODO

**Missing or non-TeX source handling:** keep an empty record rather than dropping the publication; explain the implementation here.

### 2.3 Metadata and references

Describe the paper metadata fields collected and the Semantic Scholar API fields used for crawled references. Explain rate limiting, missing API results, and retry behavior.

**Implementation:** TODO

## 3. Milestone 1 — output tables and data quality

The final dataset tables introduced by Milestone 1 are `papers` and `references`. These tables are described here for analysis; the working corpus may contain additional fields/files.

### 3.1 Expected schema

**`papers`**

| Column | Meaning |
|---|---|
| `arxiv_id` | Base arXiv ID; required |
| `title` | Paper title |
| `authors` | List of author names |
| `submission_date` | Initial submission date |
| `revised_dates` | List of revision dates |
| `venue` | Venue when available; nullable |

**`references`**

| Column | Meaning |
|---|---|
| `arxiv_id` | Citing paper's arXiv ID |
| `ref_arxiv_id` | Referenced paper's arXiv ID |
| `ref_title` | Reference title |
| `ref_authors` | List of reference authors |
| `ref_submission_date` | Reference submission date |
| `ref_s2_id` | Semantic Scholar paper ID |

The working corpus should retain all crawled reference records. The published `references` table includes references with an arXiv ID, as required by the specification.

In [1]:
# TODO: Replace these paths/column names with the group's actual outputs.
PAPERS_PATH = Path("data/processed/papers.parquet")
REFERENCES_PATH = Path("data/processed/references.parquet")

def load_table(path):
    if not path.exists():
        print(f"Not found yet: {path}")
        return pd.DataFrame()
    return pd.read_parquet(path)

papers = load_table(PAPERS_PATH)
references = load_table(REFERENCES_PATH)

print("papers:", papers.shape)
print("references:", references.shape)
display(papers.head())
display(references.head())

NameError: name 'Path' is not defined

### 3.2 Crawling coverage and success rate

Report at least:
- Number of assigned base IDs attempted
- Number of papers resolved successfully
- Number of versions expected and number downloaded successfully
- Number of papers/versions with missing or non-TeX sources
- Overall success rate, with the denominator clearly defined

**Definition used for each rate:** TODO. Keep paper-level and version-level rates separate.


In [2]:
# TODO: Load the crawl manifest/log produced by the actual crawler.
# Suggested manifest columns:
# arxiv_id, version, metadata_ok, source_download_ok, tex_or_bib_found,
# semantic_scholar_ok, error_type, elapsed_seconds, output_bytes

MANIFEST_PATH = Path("data/processed/crawl_manifest.csv")
if MANIFEST_PATH.exists():
    crawl_manifest = pd.read_csv(MANIFEST_PATH)
    display(crawl_manifest.head())
    print("Manifest rows:", len(crawl_manifest))
else:
    crawl_manifest = pd.DataFrame()
    print(f"Create the crawler manifest at {MANIFEST_PATH} to calculate coverage.")

NameError: name 'Path' is not defined

In [ ]:
# TODO: Adjust these calculations to match the manifest's real semantics.
if not crawl_manifest.empty:
    if "source_download_ok" in crawl_manifest.columns:
        print("Version-level source download success rate:",
              crawl_manifest["source_download_ok"].fillna(False).astype(bool).mean())
    if "semantic_scholar_ok" in crawl_manifest.columns:
        print("Reference metadata request success rate:",
              crawl_manifest["semantic_scholar_ok"].fillna(False).astype(bool).mean())

### 3.3 Corpus statistics and figures

Generate figures from the actual data during book compilation. Suggested analyses:
- Number of versions per publication
- Distribution of reference counts per paper
- Missingness of key metadata fields
- Average source size before and after excluding figure binaries, if both measurements were recorded

Interpret each figure in text; do not leave charts without an explanation.


In [ ]:
# TODO: Replace column names with those available in the real tables/manifest.
if not papers.empty:
    print("Paper table columns:", papers.columns.tolist())
if not references.empty:
    print("Reference table columns:", references.columns.tolist())

# Example plotting template (uncomment/adapt after confirming the real column names).
# if "ref_count" in papers.columns:
#     ax = papers["ref_count"].dropna().plot(kind="hist", bins=30)
#     ax.set_title("Distribution of references per paper")
#     ax.set_xlabel("References per paper")
#     ax.set_ylabel("Number of papers")
#     plt.tight_layout()
#     plt.show()


## 4. Milestone 1 — performance benchmark

Benchmark on the specified CPU-only Google Colab testbed. Explain whether timings are end-to-end wall time and how RAM/storage are measured.

Required report items:
- Total time for entry discovery
- Total end-to-end time to process the corpus
- Average processing time per required paper (define the denominator)
- Peak RAM and average RAM consumption
- Maximum disk storage required and final output storage size
- Average source/archive size before and after figure binaries are excluded
- Average references per paper and Semantic Scholar reference-metadata success rate

| Metric | Value | Measurement method / notes |
|---|---:|---|
| Entry discovery total time (s) | TODO | TODO |
| End-to-end crawling time (s) | TODO | TODO |
| Average time per paper (s) | TODO | TODO |
| Peak RAM (MiB/GiB) | TODO | TODO |
| Average RAM (MiB/GiB) | TODO | TODO |
| Maximum disk usage (MiB/GiB) | TODO | TODO |
| Final output size (MiB/GiB) | TODO | TODO |
| Average source size before filtering | TODO | TODO |
| Average source size after filtering | TODO | TODO |

### 4.1 Benchmark method

**Runtime measurement:** TODO  
**RAM measurement:** TODO  
**Disk usage measurement:** TODO  
**Hardware/runtime configuration:** TODO

## 5. Milestone 1 — discussion and limitations

Discuss:
- Which failure cases occurred most often?
- Were some versions unavailable or not provided as TeX?
- How complete was Semantic Scholar metadata?
- What trade-offs were made between request rate, runtime, retries, and reliability?
- What should be improved before parsing?

**Findings:** TODO

**Limitations:** TODO

# Part II — Milestone 2: Hierarchical Parsing and Standardization

## 6. Parsing methodology

### 6.1 Main compilation path

The publication root is the `.tex` file containing `\\documentclass`. Follow `\\input` and `\\include` recursively, ignoring commands commented out in LaTeX comments. Parse only files reachable from the root; ignore redundant auxiliary `.tex` files.

Explain how the implementation handles:
- Root detection and multiple possible roots
- Relative paths and omitted `.tex` extensions
- Recursive includes and include cycles
- Comments and missing included files
- Versions without TeX sources (record as empty, do not silently skip)

**Implementation:** TODO

In [ ]:
# TODO: Link or import the actual parser implementation from src/latex_parser/.
# Example:
# from src.latex_parser import find_root_tex, collect_compilation_files, parse_publication

### 6.2 Hierarchy and element extraction

Describe how the parser represents a document as a tree and assigns parent-child relationships. Required leaf granularity includes:
- Sentences from prose, without splitting at non-sentence periods in math, commands, abbreviations (e.g. `i.e.`, `et al.`, `Fig. 1`), URLs, or decimal numbers
- Block formulas (e.g. `equation` environments or `$$...$$`)
- Figures; tables are treated as a type of figure
- Itemized/enumerated lists as branching components with each item at the next level

Include acknowledgements and appendices, including unnumbered `\\section*` sections. Do not parse the references section into the hierarchy.

**Element kinds and hierarchy-level rules:** TODO

**Sentence boundary handling:** TODO

**Formula / figure / table handling:** TODO

### 6.3 Cleanup and math normalization

Explain how non-semantic LaTeX formatting is removed (examples in the specification include `\\centering`, placement options such as `[htpb]`, and `\\midrule`) while preserving semantic content. State the chosen unified representation for inline and block math.

**Cleanup rules:** TODO

**Inline math format:** TODO

**Block math format:** TODO

**Validation checks:** TODO

### 6.4 `\\bibitem` conversion and reference deduplication

Describe the programmatic conversion from `\\bibitem` definitions to standard BibTeX entries. Then explain cross-version deduplication:
- Identify entries referring to the same underlying reference, even if citation keys differ.
- Select one canonical citation key.
- Merge fields from duplicate entries instead of discarding information.
- Rewrite `\\cite{...}` commands to use the canonical key.

**Conversion method:** TODO

**Matching/canonical-key policy:** TODO

**Field union policy:** TODO

**Citation-key rewriting and tests:** TODO